# Pipeline test

Tests every source and every step, then checks the numbers of the dataset against the original files.

**How to use it**

1. Open it from the project folder, in the environment with the libraries installed, with the keys in `.env`.
2. Edit the settings in section 0.
3. Run all the cells in order. A failing check does not stop the notebook.
4. Read the table in section 16 and send it (`test_results.csv`) if something is wrong.

Every check prints one line:

- **PASS**: it works.
- **FAIL**: it does not work, or two numbers differ.
- **SKIP**: it could not run, for example a missing key. The reason is printed.

The downloads write in `output/`, as the normal workflow does. With `RUN_DOWNLOADS = False` the notebook only checks what is already saved.


## 0 - Settings and helpers


In [ ]:
ISO3 = "AFG"
YEAR = 2025
EVENTTYPE = "EQ"

RUN_DOWNLOADS = True       # False: only check what is already in output/
MAX_EVENTS = 3             # events to download in the test; None = all of them
RELIEFWEB_DAYS = 7         # days after the event; more days means more PDFs
RUN_GHSL_FULL = False      # True: every admin unit of the country, not only a sample
RUN_EMBEDDINGS = False     # True: embeddings of the ReliefWeb texts (slow, needs torch)

POP_TIF = None             # None: rasters found through RELEASE_ROOT in sources/ghsl/ghsl.py
SMOD_TIF = None


In [ ]:
import importlib
import json
import os
import platform
import re
import sys
from contextlib import contextmanager
from pathlib import Path

import pandas as pd

RESULTS = []


class Skip(Exception):
    """Raised to skip the rest of a block, with the reason."""


def record(section, name, status, detail=""):
    RESULTS.append({"section": section, "check": name, "status": status, "detail": str(detail)[:300]})
    print(f"[{status}] {section} | {name}" + (f" | {detail}" if detail != "" else ""))


def check(section, name, ok, detail=""):
    record(section, name, "PASS" if ok else "FAIL", detail)
    return bool(ok)


def same(section, name, got, expected, rel=1e-6):
    """Two numbers must match, within a relative tolerance."""
    if pd.isna(got) or pd.isna(expected):
        return check(section, name, False, f"missing value: dataset {got}, original {expected}")
    ok = abs(float(got) - float(expected)) <= rel * max(1.0, abs(float(expected)))
    return check(section, name, ok, f"dataset {got:,.6g}, original {expected:,.6g}")


@contextmanager
def block(section, name):
    """Runs a block of a test: an error becomes a FAIL and the notebook goes on."""
    try:
        yield
    except Skip as e:
        record(section, name, "SKIP", e)
    except Exception as e:
        record(section, name, "FAIL", f"{type(e).__name__}: {e}")


def need_downloads():
    if not RUN_DOWNLOADS:
        raise Skip("RUN_DOWNLOADS is False")


def need_key(*names):
    missing = [n for n in names if not os.getenv(n)]
    if missing:
        raise Skip(f"missing in .env: {', '.join(missing)}")


def country_folder():
    return Path("output") / ISO3 / str(YEAR) / "country"


def event_folders():
    return sorted(p.parent.parent for p in Path("output", ISO3, str(YEAR)).glob("*/gdacs/eventdata.json"))


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


## 1 - Environment


In [ ]:
S = "1 environment"
check(S, "project folder", Path("sources/dataset/build.py").is_file(), Path.cwd())
print("Python", platform.python_version())

check(S, "pandas 2 or newer", int(pd.__version__.split(".")[0]) >= 2, pd.__version__)
for lib in ("numpy", "pyarrow", "openpyxl", "requests", "httpx", "dotenv", "geopandas", "pyogrio", "shapely",
            "rasterio", "exactextract", "pystac_client", "dtmapi", "pdfplumber"):
    with block(S, f"library {lib}"):
        importlib.import_module(lib)
        record(S, f"library {lib}", "PASS")

with block(S, "keys"):
    from dotenv import load_dotenv
    load_dotenv()
    for key in ("IDMC_CLIENT_ID", "DTMAPI_SUBSCRIPTION_KEY", "MONTANDON_API_TOKEN"):
        record(S, f"key {key}", "PASS" if os.getenv(key) else "SKIP", "" if os.getenv(key) else "not in .env")
    hdx = os.getenv("HDX_APP_IDENTIFIER") or (os.getenv("HDX_APP_NAME") and os.getenv("HDX_APP_EMAIL"))
    record(S, "key HDX", "PASS" if hdx else "SKIP", "" if hdx else "not in .env")


## 2 - Boundaries


In [ ]:
S = "2 boundaries"
from sources.boundaries import available_levels, download_boundaries, load_admin

with block(S, "download"):
    need_downloads()
    levels = download_boundaries(ISO3)
    check(S, "download", len(levels) > 0, {k: len(v) for k, v in levels.items()})

with block(S, "saved files"):
    found = available_levels(ISO3)
    if not check(S, "levels saved", 0 in found and len(found) >= 2, found):
        raise Skip("no boundaries")
    a1, a2 = load_admin(ISO3, 1), load_admin(ISO3, 2)
    check(S, "admin 1 Pcodes unique", a1.pcode.is_unique, len(a1))
    check(S, "admin 2 Pcodes unique", a2.pcode.is_unique, len(a2))
    check(S, "every admin 2 has its admin 1", set(a2.parent_pcode) <= set(a1.pcode))
    check(S, "CRS is EPSG:4326", a2.crs.to_epsg() == 4326)
    check(S, "valid geometries", int((~a2.geometry.is_valid).sum()) == 0, f"{int((~a2.geometry.is_valid).sum())} invalid")


## 3 - GDACS, EM-DAT and EMM


In [ ]:
S = "3 gdacs"
from sources.gdacs import episode_ids, list_areas, search_events
from sources.gdacs.extract_variables import fetch_event, save_variables

with block(S, "event list"):
    need_downloads()
    events = search_events(EVENTTYPE, f"{YEAR}-01-01", f"{YEAR}-12-31", country=ISO3)
    alerts = pd.Series([f["properties"]["alertlevel"] for f in events]).value_counts().to_dict()
    check(S, "event list", len(events) > 0, f"{len(events)} events {alerts}")
    check(S, "Green events included", alerts.get("Green", 0) > 0, alerts)

    rank = {"Red": 0, "Orange": 1, "Green": 2}
    ordered = sorted(events, key=lambda f: (rank.get(f["properties"]["alertlevel"], 3), f["properties"]["eventid"]))
    chosen = ordered
    if MAX_EVENTS is not None:      # the strongest ones and one Green, which has no ShakeMap
        chosen = ordered[:MAX_EVENTS - 1] + [ordered[-1]]
    for f in chosen:
        p = f["properties"]
        with block(S, f"download {p['eventid']}"):
            folder = fetch_event(EVENTTYPE, p["eventid"])
            save_variables(folder)
            record(S, f"download {p['eventid']}", "PASS", f"{p['alertlevel']} -> {folder}")


In [ ]:
with block(S, "saved files"):
    folders = event_folders()
    if not check(S, "event folders", len(folders) > 0, len(folders)):
        raise Skip("no events")
    for d in folders:
        e = load_json(d / "gdacs" / "eventdata.json")["properties"]
        episodes = sorted((d / "episodes").glob("*"))
        problems = []
        if not (d / "gdacs" / "adaptivecard.json").is_file():
            problems.append("no card")
        if not (d / "variables.csv").is_file():
            problems.append("no variables.csv")
        if not episodes:
            problems.append("no episodes")
        for ep in episodes:
            for name in ("episodedata.json", "geometry.json"):
                if not (ep / name).is_file():
                    problems.append(f"{ep.name} no {name}")
            if not list(ep.glob("areas/*/area.geojson")):
                problems.append(f"{ep.name} no areas")
        n_impacts = len(list(d.glob("episodes/*/impacts/*.json")))
        n_versions = len(e.get("shakemap") or [])
        n_saved = len(list(d.glob("gdacs/shakemaps/*/geometry.json")))
        if n_saved < n_versions:
            problems.append(f"ShakeMap polygons {n_saved} of {n_versions}")
        check(S, f"files of {d.name}", not problems, "; ".join(problems) or f"{len(episodes)} episodes, {n_impacts} impact runs, {n_versions} ShakeMaps")


## 4 - ReliefWeb


In [ ]:
S = "4 reliefweb"
with block(S, "download"):
    need_downloads()
    from sources.reliefweb import scrape_reliefweb
    for d in event_folders()[:MAX_EVENTS]:
        try:
            r = scrape_reliefweb(None, d, days_after=RELIEFWEB_DAYS)
            record(S, f"reports of {d.name}", "PASS", f"{r.get('verdict')}: {r.get('n_reports')} reports, {r.get('n_pdfs')} PDFs")
        except ValueError as e:         # no GLIDE: normal for small events
            record(S, f"reports of {d.name}", "SKIP", e)

with block(S, "texts"):
    if not RUN_EMBEDDINGS:
        raise Skip("RUN_EMBEDDINGS is False")
    from sources.reliefweb.embed import embed_texts, load_model
    from sources.reliefweb.extract import extract_texts
    model = load_model()
    for d in event_folders():
        f = d / "reliefweb" / "reliefweb.json"
        if f.is_file() and load_json(f).get("n_reports"):
            extract_texts(d)
            embed_texts(d, model=model)
            record(S, f"texts of {d.name}", "PASS")


## 5 - INFORM


In [ ]:
S = "5 inform"
with block(S, "download"):
    need_downloads()
    from sources.inform import build_country_table
    t = build_country_table(ISO3, YEAR)
    check(S, "indicators found", int(t.matched.sum()) >= 110, f"{int(t.matched.sum())} of {len(t)}")
    check(S, "composite score found", t.loc[t.indicator_id == "INFORM", "value"].notna().all())
    check(S, "validity years", int(t.validity_year.notna().sum()) > 0, int(t.validity_year.notna().sum()))


## 6 - IOM DTM


In [ ]:
S = "6 iom"
with block(S, "download"):
    need_downloads()
    need_key("DTMAPI_SUBSCRIPTION_KEY")
    from sources.iom.iom import download_dtm
    res = download_dtm(ISO3, f"{YEAR}-01-01", f"{YEAR}-12-31", admin_levels=[0, 1, 2])
    check(S, "download", len(res) > 0, {k: len(v) for k, v in res.items()})


## 7 - IDMC


In [ ]:
S = "7 idmc"
with block(S, "download"):
    need_downloads()
    need_key("IDMC_CLIENT_ID")
    from sources.idmc.gidd import download_gidd_country
    from sources.idmc.numbers import list_events
    download_gidd_country(ISO3, YEAR, YEAR)
    res = list_events(ISO3, YEAR, YEAR)
    check(S, "events", len(res["events"]) > 0, len(res["events"]))
    check(S, "IDU figures", len(res["idu_figures"]) > 0, len(res["idu_figures"]))
    check(S, "GIDD figures", len(res["gidd_figures"]) > 0, len(res["gidd_figures"]))


## 8 - HDX HAPI


In [ ]:
S = "8 hdx"
with block(S, "download"):
    need_downloads()
    if not (os.getenv("HDX_APP_IDENTIFIER") or (os.getenv("HDX_APP_NAME") and os.getenv("HDX_APP_EMAIL"))):
        raise Skip("missing in .env: HDX_APP_NAME and HDX_APP_EMAIL")
    import sources.hdx as hdx
    for name in ("idps", "humanitarian_needs", "refugees", "returnees", "food_security", "food_prices", "poverty_rate"):
        with block(S, name):
            df = getattr(hdx, f"download_{name}")(ISO3, f"{YEAR}-01-01", f"{YEAR}-12-31", verbose=False)
            # zero rows is not always an error: HAPI does not cover every dataset for every country
            record(S, name, "PASS" if len(df) else "SKIP", f"{len(df)} rows, {df.attrs.get('resolved_window')}")


## 9 - GHSL by admin unit

Population is the sum of the people in the cells that touch the unit. With `RUN_GHSL_FULL = False` only the units of one province are computed, and nothing is saved.


In [ ]:
S = "9 ghsl admin"
GHSL_SAMPLE = None


def rasters():
    from sources import ghsl
    if POP_TIF and SMOD_TIF:
        return SMOD_TIF, POP_TIF
    if not ghsl.ghsl.RELEASE_ROOT.is_dir():
        raise Skip(f"rasters not found: {ghsl.ghsl.RELEASE_ROOT}")
    return ghsl.get_smod_and_pop(YEAR)


with block(S, "calculation"):
    from sources.boundaries import load_admin
    from sources.ghsl.admin import admin_stats
    smod_tif, pop_tif = rasters()
    a2 = load_admin(ISO3, 2)
    first = sorted(a2.parent_pcode.unique())[0]
    sample = sorted(a2[a2.parent_pcode == first].pcode)
    GHSL_SAMPLE = admin_stats(ISO3, YEAR, pop_tif=pop_tif, smod_tif=smod_tif, pcodes=sample, save=False)
    t1, t2 = GHSL_SAMPLE[1], GHSL_SAMPLE[2]
    check(S, "one row per unit", len(t2) == len(sample), f"{len(t2)} units of {first}")
    check(S, "border cells count twice", t2.pop_ghsl.sum() >= t1.pop_ghsl.sum(), f"admin 2 {t2.pop_ghsl.sum():,.0f}, admin 1 {t1.pop_ghsl.sum():,.0f}")
    classes = [c for c in t1.columns if re.fullmatch(r"pop_ghsl_\d+", c)]
    same(S, "classes add up to the total", t1[classes].sum(axis=1).iloc[0], t1.pop_ghsl.iloc[0])


In [ ]:
with block(S, "independent check"):
    if GHSL_SAMPLE is None:
        raise Skip("no sample")
    import rasterio
    from rasterio.features import rasterize
    from rasterio.windows import from_bounds
    smod_tif, pop_tif = rasters()

    def touching(geom, tif):
        """Values of all the cells touching a geometry, with rasterio instead of exactextract."""
        with rasterio.open(tif) as src:
            b = geom.bounds
            w = from_bounds(b[0] - 1000, b[1] - 1000, b[2] + 1000, b[3] + 1000, src.transform).round_offsets().round_lengths()
            data, tr, nodata = src.read(1, window=w), src.window_transform(w), src.nodata
        values = data[rasterize([geom], out_shape=data.shape, transform=tr, all_touched=True).astype(bool)]
        return values[values != nodata] if nodata is not None else values

    units = load_admin(ISO3, 2).to_crs("ESRI:54009").set_index("pcode")
    for _, row in GHSL_SAMPLE[2].head(3).iterrows():
        pop = touching(units.loc[row.pcode].geometry, pop_tif)
        cells = touching(units.loc[row.pcode].geometry, smod_tif)
        same(S, f"population of {row.pcode}", row.pop_ghsl, float(pop[pop > 0].sum()), rel=0.005)
        same(S, f"cells of {row.pcode}", row.filter(like="smod_cells").sum(), len(cells), rel=0.005)


In [ ]:
with block(S, "full run"):
    if not RUN_GHSL_FULL:
        raise Skip("RUN_GHSL_FULL is False")
    smod_tif, pop_tif = rasters()
    res = admin_stats(ISO3, YEAR, pop_tif=pop_tif, smod_tif=smod_tif)
    check(S, "full run", len(res) > 0, {k: len(v) for k, v in res.items()})


## 10 - GHSL by location


In [ ]:
S = "10 ghsl location"
with block(S, "one area of one event"):
    from sources import ghsl
    from sources.utils.geo import dissolve_features, read_features, to_raster_crs
    smod_tif, pop_tif = rasters()
    with_bands = [d for d in event_folders() if list(d.glob("episodes/*/areas/Intensity_*/area.geojson"))]
    if not with_bands:
        raise Skip("no event with bands")
    d = with_bands[-1]
    main = episode_ids(d / "gdacs" / "eventdata.json")["episodeid"]
    areas = list_areas(d / "episodes" / str(main))
    label = sorted(a for a in areas if a.startswith("Intensity"))[-1]      # the smallest band
    gdf = to_raster_crs(dissolve_features(read_features(areas[label])), pop_tif)
    pop_cells, pop_area = ghsl.extract_pop(gdf, pop_tif, areas[label].parent)
    smod_cells, smod_area = ghsl.extract_smod(gdf, smod_tif, areas[label].parent)
    same(S, f"{d.name} {label}: area total and cells", pop_area.pop_sum.sum(), pop_cells.pop_covered.sum(), rel=0.001)
    check(S, "SMOD cells", len(smod_cells) > 0, len(smod_cells))


## 11 - Montandon

These functions have never run on real records: this is the first real test. If something fails, send the message.


In [ ]:
S = "11 montandon"
with block(S, "connection"):
    need_key("MONTANDON_API_TOKEN")
    from sources import montandon
    client = montandon.connect()
    collections = montandon.list_collections(client)
    check(S, "collections", len(collections["events"]) > 0, {k: len(v) for k, v in collections.items()})

    start, end = f"{YEAR}-01-01T00:00:00Z", f"{YEAR}-12-31T23:59:59Z"
    with block(S, "events of all sources"):
        events = montandon.search_all_events(client, ISO3, start, end, montandon.hazard_codes_for_type(EVENTTYPE))
        table = montandon.events_table(events)
        check(S, "events of all sources", len(table) > 0, table.collection.value_counts().to_dict())
    with block(S, "impacts by type"):
        rows = montandon.impacts_table(montandon.impacts_by_type(client, ISO3, start, end, types=["displaced_internal"]))
        record(S, "impacts by type", "PASS", rows.collection.value_counts().to_dict())

    for d in event_folders()[:MAX_EVENTS]:
        with block(S, f"event {d.name}"):
            need_downloads()
            eventid = d.name.split("_")[1]
            saved = montandon.fetch_event(client, eventid, d, EVENTTYPE)
            record(S, f"event {d.name}", "PASS" if saved else "SKIP", list(saved) if saved else "not in Montandon")


## 12 - Sentinel-2


In [ ]:
S = "12 sentinel2"
with block(S, "catalog search"):
    try:
        from sources import sentinel2
        importlib.import_module("planetary_computer")
    except ModuleNotFoundError as e:
        raise Skip(f"optional library missing: {e.name}")
    folders = [d for d in event_folders() if list(d.glob("episodes/*/areas/Intensity_*/area.geojson"))]
    if not folders:
        raise Skip("no event with bands")
    d = folders[-1]
    date = load_json(d / "gdacs" / "eventdata.json")["properties"]["fromdate"][:10]
    area = sorted(d.glob("episodes/*/areas/Intensity_*/area.geojson"))[-1]
    items = sentinel2.search_items(sentinel2.open_catalog(), sentinel2.aoi_from_geojson(area), date, span_days=15)
    check(S, "images found", len(items) > 0, len(items))


## 13 - Build the dataset


In [ ]:
S = "13 build"
from sources.dataset import access
from sources.dataset.build import build

with block(S, "build"):
    problems = build(iso3=[ISO3])
    check(S, "no problems", not problems, problems[:5])


## 14 - Tables


In [ ]:
S = "14 tables"
LEVELS = ("admin0", "admin1", "admin2", "event", "episode", "location")
KEYS = {"admin0": ["iso3"], "admin1": ["admin1_pcode"], "admin2": ["admin2_pcode"], "event": ["event_id"],
        "episode": ["event_id", "episode_id"], "location": ["event_id", "episode_id", "location_id"]}
ID_COLUMNS = {"iso3", "year", "admin1_pcode", "admin1_name", "admin2_pcode", "admin2_name", "event_id", "episode_id",
              "location_id", "hazard_type"}
T = {}

with block(S, "tables"):
    columns = access.columns(ISO3, YEAR)
    for level in LEVELS:
        with block(S, level):
            t = access.table(ISO3, YEAR, level)
            T[level] = t
            check(S, f"{level}: one row per unit", not t.duplicated(KEYS[level]).any(), f"{t.shape[0]} rows, {t.shape[1]} columns")
            described = set(columns[columns.level == level].column)
            extra = sorted(set(t.columns) - ID_COLUMNS - described)
            check(S, f"{level}: every column is in columns.csv", not extra, extra[:5])
            empty = [c for c in t.columns if t[c].isna().all()]
            check(S, f"{level}: no empty column", not empty, empty[:5])
    check(S, "validity of the columns", columns.valid_from.notna().all(), f"{len(columns)} columns")
    if "admin2" in T and available_levels(ISO3):
        check(S, "admin 2 rows equal the boundaries", len(T["admin2"]) == len(load_admin(ISO3, 2)), len(T["admin2"]))
    display(columns.groupby(["level", "source"]).size().rename("columns").reset_index())


In [ ]:
with block(S, "access functions"):
    v = access.list_variables("event")
    check(S, "list_variables", len(v) > 0, len(v))
    g = access.get("hazard_magnitude", level="event", iso3=ISO3)
    check(S, "get", len(g) > 0, len(g))
    d = access.wide(level="admin0", iso3=ISO3, date=f"{YEAR}-12-31", variables=["pop_hno", "disp_flow"])
    check(S, "wide", len(d) == 1 or len(d) == 0, d.shape)
    links = access.links()
    check(S, "links", len(links) > 0, links.source.value_counts().to_dict())


In [ ]:
with block(S, "as_of and nowcast"):
    versions = access.get("shakemap_max_mmi", level="event", iso3=ISO3)
    if versions.empty:
        raise Skip("no ShakeMap versions")
    event = versions.groupby("event_id").size().idxmax()
    rows = versions[versions.event_id == event].sort_values("known_at")
    first, last = rows.iloc[0], rows.iloc[-1]
    early = first.known_at + pd.Timedelta(minutes=1)
    got = access.as_of("shakemap_max_mmi", level="event", event_id=event, date=early, known_by=early)
    check(S, "known_by gives the first version", got.source_version.iloc[0] == first.source_version, f"version {got.source_version.iloc[0]} of {event}")
    got = access.as_of("shakemap_max_mmi", level="event", event_id=event, date=last.known_at, known_by=last.known_at)
    check(S, "later gives the last version", got.source_version.iloc[0] == last.source_version, f"version {got.source_version.iloc[0]}")


## 15 - Are the numbers right?

Every check goes back to the **original file** and recomputes the value without using the extractors. Three kinds of check:

- **15a, dataset against originals:** the value in the table is the one the source gave.
- **15b, consistency:** totals, dates and signs make sense.
- **15c, sources against each other:** different sources should be close. Here a difference is a warning to read, not an error.
- **15d, by hand:** a random sample to compare on the website of the source.


In [ ]:
S = "15a originals"
events_t = T.get("event", pd.DataFrame())


def row(table, event_id):
    r = table[table.event_id == event_id]
    return r.iloc[0] if len(r) else None


with block(S, "GDACS"):
    for d in event_folders():
        r = row(events_t, d.name)
        p = load_json(d / "gdacs" / "eventdata.json")
        if r is None:
            check(S, f"{d.name} in event table", False)
            continue
        same(S, f"{d.name} severity", r.hazard_severity, p["properties"]["severitydata"]["severity"])
        lon, lat = p["geometry"]["coordinates"][:2]
        lat_col = "hazard_lat_gdacs" if "hazard_lat_gdacs" in r else "hazard_lat"
        lon_col = "hazard_lon_gdacs" if "hazard_lon_gdacs" in r else "hazard_lon"
        same(S, f"{d.name} latitude", r[lat_col], lat)
        same(S, f"{d.name} longitude", r[lon_col], lon)
        check(S, f"{d.name} alert", r.alert_level == p["properties"]["alertlevel"], r.alert_level)

with block(S, "EM-DAT"):
    done = 0
    for d in event_folders():
        f = d / "gdacs" / "emdat.json"
        rec = load_json(f) if f.is_file() else []
        rec = rec[0] if rec else None
        r = row(events_t, d.name)
        if rec is None or r is None:
            continue
        for column, field in (("impact_deaths", "totaldeaths"), ("impact_total_affected", "totalaffected"), ("impact_homeless", "numhomeless")):
            if rec.get(field) is not None:
                same(S, f"{d.name} {field}", r[column], rec[field])
                done += 1
    if not done:
        raise Skip("no EM-DAT record")

with block(S, "impact runs"):
    from sources.gdacs.extract_variables import impact_scalars
    ep_t = T.get("episode", pd.DataFrame())
    files = sorted(Path("output", ISO3, str(YEAR)).glob("*/episodes/*/impacts/shake_preliminary_*.json"))
    if not files:
        raise Skip("no impact runs")
    for f in files[:5]:
        event, episode = f.parents[3].name, f.parents[1].name
        raw = {n: v for _, n, v in impact_scalars(load_json(f))}
        r = ep_t[(ep_t.event_id == event) & (ep_t.episode_id == episode)]
        if "PopulationOverVII" in raw and len(r):
            same(S, f"{event} {episode} population over VII", r["populationovervii_shake_preliminary"].iloc[0], raw["PopulationOverVII"])


In [ ]:
with block(S, "IOM DTM"):
    files = sorted(country_folder().glob("iom/dtm_admin2_*.json"))
    files = [f for f in files if not f.name.endswith(".meta.json")]
    if not files:
        raise Skip("no DTM file")
    raw = pd.concat([pd.DataFrame(load_json(f)) for f in files]).drop_duplicates("id")
    raw = raw[raw.reportingDate == raw.reportingDate.max()]
    same(S, "IDPs, sum of admin 2", T["admin2"].idp_stock_dtm.sum(), raw.numPresentIdpInd.sum())
    same(S, "IDPs, admin 0", T["admin0"].idp_stock_dtm.iloc[0], raw.numPresentIdpInd.sum())

with block(S, "HDX needs"):
    files = sorted(country_folder().glob("hdx/humanitarian_needs/*.parquet"))
    if not files:
        raise Skip("no file")
    raw = pd.concat([pd.read_parquet(f) for f in files]).drop_duplicates()
    raw = raw[raw.reference_period_start.str[:4] == str(YEAR)]
    pick = raw[(raw.sector_name == "Intersectoral") & (raw.category == "total")]
    for column, status in (("needs_pin", "INN"), ("needs_targeted", "TGT"), ("pop_hno", "all")):
        same(S, f"{column}, admin 0", T["admin0"][column].iloc[0], pick[(pick.admin_level == 0) & (pick.population_status == status)].population.iloc[0])
        same(S, f"{column}, sum of admin 2", T["admin2"][column].sum(), pick[(pick.admin_level == 2) & (pick.population_status == status)].population.sum())

with block(S, "HDX refugees"):
    files = sorted(country_folder().glob("hdx/refugees/*.parquet"))
    if not files:
        raise Skip("no file")
    raw = pd.concat([pd.read_parquet(f) for f in files]).drop_duplicates()
    raw = raw[(raw.gender == "all") & (raw.age_range == "all") & raw.population_group.isin(["REF", "ASY", "OIP"])]
    raw = raw[raw.reference_period_start == raw.reference_period_start.max()]
    same(S, "refugees hosted", T["admin0"].refugees_hosted.iloc[0],
         raw[(raw.asylum_location_code == ISO3) & (raw.origin_location_code != ISO3)].population.sum())
    same(S, "refugees abroad", T["admin0"].refugees_abroad.iloc[0],
         raw[(raw.origin_location_code == ISO3) & (raw.asylum_location_code != ISO3)].population.sum())

with block(S, "GIDD country"):
    files = [f for f in sorted(country_folder().glob("idmc/gidd_country_*.json")) if not f.name.endswith(".meta.json")]
    if not files:
        raise Skip("no file")
    raw = [r for f in files for r in load_json(f) if r["year"] == YEAR]
    if not raw:
        raise Skip(f"GIDD has no {YEAR}")
    same(S, "disaster flow", T["admin0"].disp_flow_disaster_gidd.iloc[0], raw[0]["disaster_new_displacement"])
    same(S, "conflict stock", T["admin0"].disp_stock_conflict_gidd.iloc[0], raw[0]["conflict_total_displacement"])

with block(S, "IDU events"):
    files = sorted(country_folder().glob("idmc/events_overview_*.parquet"))
    if not files:
        raise Skip("no file")
    raw = pd.read_parquet(files[-1]).set_index("event_id")
    links = access.links()
    primary = links[(links.source == "idmc") & links.primary]
    primary = primary[primary.event_id.isin(events_t.event_id)]
    if primary.empty:
        raise Skip("no linked event")
    for r in primary.itertuples():
        if int(r.source_event_id) in raw.index:
            same(S, f"{r.event_id} from IDMC {r.source_event_id}", row(events_t, r.event_id).disp_flow_idu, raw.loc[int(r.source_event_id), "idu_flows"])

with block(S, "INFORM"):
    from sources.dataset.registry import inform_variable
    files = sorted(country_folder().glob("inform_*.csv"))
    if not files:
        raise Skip("no file")
    raw = pd.read_csv(files[-1])
    raw = raw[raw.matched & raw.value.notna()]
    a0 = T["admin0"].iloc[0]
    wrong = [n for n, v in zip(raw.indicator_name, raw.value) if abs(a0[inform_variable(n)] - v) > 1e-9 * max(1, abs(v))]
    check(S, "every indicator", not wrong, f"{len(raw) - len(wrong)} of {len(raw)} equal {wrong[:3]}")


In [ ]:
with block(S, "GHSL location and grid"):
    loc = T.get("location", pd.DataFrame())
    if "pop_ghsl" not in loc or loc.pop_ghsl.notna().sum() == 0:
        raise Skip("no GHSL location")
    r = loc[loc.pop_ghsl.notna()].iloc[0]
    g = access.grid(event_id=r.event_id, resolution_m=100, episode_id=r.episode_id)
    g = g[g.location_id == r.location_id]
    same(S, f"{r.event_id} {r.location_id}, cells", g.pop_ghsl.sum(), r.pop_ghsl)
    same(S, f"{r.event_id} {r.location_id}, covered", g.pop_ghsl_covered.sum(), r.pop_ghsl_covered, rel=0.001)
    if "pop_ghsl_rural" in r and pd.notna(r.pop_ghsl_rural):
        same(S, "rural plus urban", r.pop_ghsl_rural + r.pop_ghsl_urban, r.pop_ghsl, rel=0.001)

with block(S, "GHSL admin"):
    files = sorted(country_folder().glob("ghsl/admin2/ghsl_E*.parquet"))
    if not files:
        raise Skip("no GHSL admin table: run python -m sources.ghsl.admin")
    raw = pd.read_parquet(files[-1]).set_index("pcode")
    t = T["admin2"].set_index("admin2_pcode")
    same(S, "population, sum of admin 2", t.pop_ghsl.sum(), raw.pop_ghsl.sum())
    same(S, "urban centre cells, sum of admin 2", t.smod_cells_30.sum(), raw.smod_cells_30.sum())


In [ ]:
S = "15b consistency"
with block(S, "long tables"):
    frames = {lv: access.load(lv) for lv in LEVELS}
    for lv, df in frames.items():
        df = df[df.iso3 == ISO3]
        backwards = df[df.valid_to < df.valid_from]
        check(S, f"{lv}: validity runs forward", backwards.empty, f"{len(backwards)} of {len(df)} rows")
        people = df[df.unit == "people"]
        check(S, f"{lv}: no negative people", int((people.value < 0).sum()) == 0, f"{len(people)} rows")
        check(S, f"{lv}: every row has source and date", df.source.notna().all() and df.valid_from.notna().all())

with block(S, "shares"):
    for level, t in T.items():
        shares = [c for c in t.columns if "share" in c and pd.api.types.is_numeric_dtype(t[c])]
        bad = [c for c in shares if ((t[c] < 0) | (t[c] > 1)).any()]
        check(S, f"{level}: shares between 0 and 1", not bad, bad[:4])

with block(S, "totals"):
    a0, a1, a2 = T["admin0"], T["admin1"], T["admin2"]
    if "pop_hno" in a2:
        same(S, "HNO population: admin 2 and admin 0", a2.pop_hno.sum(), a0.pop_hno.iloc[0], rel=0.001)
    if "pop_ghsl" not in a2:
        raise Skip("no GHSL admin")
    check(S, "GHSL: admin 2 >= admin 1 >= admin 0", a2.pop_ghsl.sum() >= a1.pop_ghsl.sum() >= a0.pop_ghsl.iloc[0],
          f"{a2.pop_ghsl.sum():,.0f}, {a1.pop_ghsl.sum():,.0f}, {a0.pop_ghsl.iloc[0]:,.0f}")
    check(S, "GHSL: border effect under 3%", a2.pop_ghsl.sum() / a0.pop_ghsl.iloc[0] < 1.03)

with block(S, "events"):
    e = T["event"]
    start = pd.to_datetime(access.get("event_name", level="event", iso3=ISO3, source="gdacs").valid_from)
    check(S, "GDACS events start in the year", (start.dt.year == YEAR).all(), f"{len(start)} events")
    links = access.links()
    idmc = links[links.source == "idmc"]
    if len(idmc):
        one = idmc[idmc.primary].groupby("source_event_id").size().eq(1).all()
        check(S, "each IDMC event has one main folder", one, f"{idmc.source_event_id.nunique()} IDMC events, {len(idmc)} links")


In [ ]:
S = "15c between sources"
print("Differences here are worth reading: sources measure different things at different times.")
with block(S, "population"):
    a0 = T["admin0"].iloc[0]
    if "pop_ghsl" in a0 and "pop_hno" in a0:
        ratio = a0.pop_ghsl / a0.pop_hno
        check(S, "GHSL and HNO within 25%", 0.75 < ratio < 1.25, f"GHSL {a0.pop_ghsl:,.0f}, HNO {a0.pop_hno:,.0f}, ratio {ratio:.2f}")
    else:
        raise Skip("needs GHSL admin and HNO")

with block(S, "events"):
    e = T["event"]
    if "hazard_magnitude_emdat" in e:
        both = e.dropna(subset=["hazard_magnitude_gdacs", "hazard_magnitude_emdat"])
        gap = (both.hazard_magnitude_gdacs - both.hazard_magnitude_emdat).abs()
        check(S, "GDACS and EM-DAT magnitude within 0.5", (gap <= 0.5).all(), f"{len(both)} events, largest gap {gap.max() if len(gap) else 0}")
    wanted = [c for c in ("event_id", "alert_level", "hazard_magnitude_gdacs", "impact_deaths", "impact_homeless", "disp_flow_idu",
                          "disp_flow_gidd", "montandon_displaced_internal_idmc_idu", "montandon_death_emdat", "exposure_pop_mmi_ge_7") if c in e]
    with_impact = [c for c in wanted if c in ("impact_deaths", "impact_homeless", "disp_flow_idu", "disp_flow_gidd")]
    display(e[wanted].dropna(subset=with_impact, how="all"))


In [ ]:
S = "15d by hand"
with block(S, "random sample"):
    long = pd.concat([access.load(lv) for lv in ("admin0", "admin2", "event")], ignore_index=True)
    long = long[(long.iso3 == ISO3) & long.value.notna()]
    picked = long.groupby("source", group_keys=False).sample(n=1, random_state=YEAR)
    print("Open source_file, or the website of the source, and compare the value by hand.")
    display(picked[["level", "source", "variable", "value", "unit", "admin2_name", "event_id", "valid_from", "source_file"]])
    for d in event_folders()[:MAX_EVENTS]:
        print(load_json(d / "gdacs" / "eventdata.json")["properties"]["url"]["report"])
    record(S, "random sample", "PASS", f"{len(picked)} values to compare by hand")


## 16 - Results


In [ ]:
results = pd.DataFrame(RESULTS)
results.to_csv("test_results.csv", index=False, encoding="utf-8-sig")
display(results.groupby(["section", "status"]).size().unstack(fill_value=0))
print(results.status.value_counts().to_dict())
display(results[results.status == "FAIL"])
display(results[results.status == "SKIP"])
